In [1]:
import sys
import os
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

sys.path.append(os.path.abspath('..'))

from src.models import MLP
from src.train import treinar
from src.metrics import avaliar_regressao, avaliar_classificacao_multiclasse

def fixar_sementes(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

os.makedirs('../results', exist_ok=True)

In [2]:
def preparar_regressao(batch_size=32):
    target_col = 'Mental_Wellbeing_Score'
    df_tr = pd.read_csv('../data/processed/train_regression.csv')
    df_te = pd.read_csv('../data/processed/test_regression.csv')

    X_full, y_full = df_tr.drop(columns=[target_col]).values, df_tr[target_col].values
    X_test, y_test = df_te.drop(columns=[target_col]).values, df_te[target_col].values

    X_train, X_val, y_train, y_val = train_test_split(X_full, y_full, test_size=0.2, random_state=42)

    sc_X = StandardScaler()
    X_tr_sc = sc_X.fit_transform(X_train)
    X_va_sc = sc_X.transform(X_val)
    X_te_sc = sc_X.transform(X_test)

    sc_y = StandardScaler()
    y_tr_sc = sc_y.fit_transform(y_train.reshape(-1, 1)).flatten()
    y_va_sc = sc_y.transform(y_val.reshape(-1, 1)).flatten()

    to_t = lambda X, y: (torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32).unsqueeze(1))
    dl_tr = DataLoader(TensorDataset(*to_t(X_tr_sc, y_tr_sc)), batch_size=batch_size, shuffle=True)
    dl_va = DataLoader(TensorDataset(*to_t(X_va_sc, y_va_sc)), batch_size=batch_size, shuffle=False)
    X_te_t = torch.tensor(X_te_sc, dtype=torch.float32)

    return dl_tr, dl_va, X_te_t, y_test, sc_X, sc_y, X_train.shape[1]

def preparar_classificacao(batch_size=32):
    target_col = 'Addiction_Level'
    df_tr = pd.read_csv('../data/processed/train_classification.csv')
    df_te = pd.read_csv('../data/processed/test_classification.csv')

    X_full, y_full = df_tr.drop(columns=[target_col]).values, df_tr[target_col].values
    X_test, y_test = df_te.drop(columns=[target_col]).values, df_te[target_col].values

    X_train, X_val, y_train, y_val = train_test_split(
        X_full, y_full, test_size=0.2, random_state=42, stratify=y_full
    )

    sc_X = StandardScaler()
    X_tr_sc = sc_X.fit_transform(X_train)
    X_va_sc = sc_X.transform(X_val)
    X_te_sc = sc_X.transform(X_test)

    to_t = lambda X, y: (torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long))
    dl_tr = DataLoader(TensorDataset(*to_t(X_tr_sc, y_train)), batch_size=batch_size, shuffle=True)
    dl_va = DataLoader(TensorDataset(*to_t(X_va_sc, y_val)), batch_size=batch_size, shuffle=False)
    X_te_t = torch.tensor(X_te_sc, dtype=torch.float32)

    return dl_tr, dl_va, X_te_t, y_test, sc_X, X_train.shape[1]

print("Funções de preparação prontas.")

Funções de preparação prontas.


In [3]:
# Grade de experimentos variando um fator por vez a partir do baseline
experimentos = [
    {"nome": "Baseline [32,32] ReLU Adam 1e-3", "hidden": [32, 32], "activation": "ReLU", "dropout": 0.0, "opt": "Adam", "lr": 1e-3},
    # Arquitetura (largura e profundidade)
    {"nome": "Arq: [16]", "hidden": [16], "activation": "ReLU", "dropout": 0.0, "opt": "Adam", "lr": 1e-3},
    {"nome": "Arq: [64, 64]", "hidden": [64, 64], "activation": "ReLU", "dropout": 0.0, "opt": "Adam", "lr": 1e-3},
    {"nome": "Arq: [128, 64, 32]", "hidden": [128, 64, 32], "activation": "ReLU", "dropout": 0.0, "opt": "Adam", "lr": 1e-3},
    # Função de ativação
    {"nome": "Ativ: Tanh", "hidden": [32, 32], "activation": "Tanh", "dropout": 0.0, "opt": "Adam", "lr": 1e-3},
    {"nome": "Ativ: GELU", "hidden": [32, 32], "activation": "GELU", "dropout": 0.0, "opt": "Adam", "lr": 1e-3},
    # Regularização (Dropout)
    {"nome": "Dropout: 0.2", "hidden": [32, 32], "activation": "ReLU", "dropout": 0.2, "opt": "Adam", "lr": 1e-3},
    {"nome": "Dropout: 0.4", "hidden": [32, 32], "activation": "ReLU", "dropout": 0.4, "opt": "Adam", "lr": 1e-3},
    # Otimizador e Taxa de Aprendizado
    {"nome": "LR: 1e-2 (Adam)", "hidden": [32, 32], "activation": "ReLU", "dropout": 0.0, "opt": "Adam", "lr": 1e-2},
    {"nome": "LR: 1e-4 (Adam)", "hidden": [32, 32], "activation": "ReLU", "dropout": 0.0, "opt": "Adam", "lr": 1e-4},
    {"nome": "Opt: SGD (lr=1e-2)", "hidden": [32, 32], "activation": "ReLU", "dropout": 0.0, "opt": "SGD", "lr": 1e-2},
]

def criar_otimizador(modelo, opt_nome, lr):
    if opt_nome == "Adam":
        return torch.optim.Adam(modelo.parameters(), lr=lr)
    elif opt_nome == "SGD":
        return torch.optim.SGD(modelo.parameters(), lr=lr, momentum=0.9)
    raise ValueError(f"Otimizador desconhecido: {opt_nome}")

def rodar_ablation(tarefa="regression", sementes=[42, 100, 2024]):
    resultados = []
    if tarefa == "regression":
        dl_tr, dl_va, X_te_t, y_test, sc_X, sc_y, in_dim = preparar_regressao()
        out_dim = 1
        loss_fn = nn.MSELoss()
    else:
        dl_tr, dl_va, X_te_t, y_test, sc_X, in_dim = preparar_classificacao()
        out_dim = 4
        loss_fn = nn.CrossEntropyLoss()

    for exp in experimentos:
        metricas_sementes = []
        tempos = []

        for seed in sementes:
            fixar_sementes(seed)
            modelo = MLP(
                in_dim=in_dim,
                out_dim=out_dim,
                hidden=exp["hidden"],
                activation=exp["activation"],
                dropout=exp["dropout"]
            )
            otim = criar_otimizador(modelo, exp["opt"], exp["lr"])

            t0 = time.perf_counter()
            treinar(modelo, dl_tr, dl_va, loss_fn, otim, n_epocas=200, paciencia=20)
            tempos.append(time.perf_counter() - t0)

            modelo.eval()
            with torch.inference_mode():
                saidas = modelo(X_te_t)
                if tarefa == "regression":
                    preds = (saidas.numpy() * sc_y.scale_[0]) + sc_y.mean_[0]
                    m = avaliar_regressao(y_test, preds)
                    metricas_sementes.append((m["RMSE"], m["MAE"], m["R2"]))
                else:
                    probs = torch.softmax(saidas, dim=1).numpy()
                    m = avaliar_classificacao_multiclasse(y_test, probs)
                    metricas_sementes.append((m["F1_macro"], m["Accuracy"], m["ROC_AUC_macro"]))

        arr = np.array(metricas_sementes)
        n_params = sum(p.numel() for p in modelo.parameters() if p.requires_grad)

        if tarefa == "regression":
            resultados.append({
                "Experimento": exp["nome"],
                "Params": n_params,
                "RMSE_media": arr[:, 0].mean(),
                "RMSE_std": arr[:, 0].std(),
                "MAE_media": arr[:, 1].mean(),
                "R2_media": arr[:, 2].mean(),
                "Tempo_medio_s": np.mean(tempos)
            })
        else:
            resultados.append({
                "Experimento": exp["nome"],
                "Params": n_params,
                "F1_macro_media": arr[:, 0].mean(),
                "F1_macro_std": arr[:, 0].std(),
                "Acc_media": arr[:, 1].mean(),
                "ROC_AUC_media": arr[:, 2].mean(),
                "Tempo_medio_s": np.mean(tempos)
            })

    return pd.DataFrame(resultados)

In [4]:
print("Rodando Ablation de Regressão (3 sementes por configuração)...")
df_res_reg = rodar_ablation("regression")
df_res_reg = df_res_reg.sort_values("RMSE_media", ascending=True).reset_index(drop=True)
df_res_reg.to_csv("../results/ablation_regressao.csv", index=False)
display(df_res_reg)

Rodando Ablation de Regressão (3 sementes por configuração)...


,Experimento,Params,RMSE_media,RMSE_std,MAE_media,R2_media,Tempo_medio_s
0,Dropout: 0.2,2049,0.494109,0.010762,0.390550,0.772424,7.308421
1,Dropout: 0.4,2049,0.509253,0.011497,0.403351,0.758252,7.356303
2,Arq: [16],497,0.527941,0.029708,0.418405,0.739494,4.634980
3,Opt: SGD (lr=1e-2),2049,0.530300,0.020061,0.417543,0.737615,1.281297
4,Ativ: GELU,2049,0.534201,0.017822,0.416232,0.733826,3.834434
5,LR: 1e-2 (Adam),2049,0.540432,0.023823,0.424890,0.727354,1.242362
6,"Baseline [32,32] ReLU Adam 1e-3",2049,0.552971,0.027628,0.434272,0.714397,2.987390
7,LR: 1e-4 (Adam),2049,0.570375,0.002716,0.452756,0.696886,7.456120
8,Ativ: Tanh,2049,0.572935,0.000458,0.460727,0.694167,1.342553
9,"Arq: [64, 64]",6145,0.581897,0.004010,0.459462,0.684509,1.790527


In [5]:
print("Rodando Ablation de Classificação (3 sementes por configuração)...")
df_res_clf = rodar_ablation("classification")
df_res_clf = df_res_clf.sort_values("F1_macro_media", ascending=False).reset_index(drop=True)
df_res_clf.to_csv("../results/ablation_classificacao.csv", index=False)
display(df_res_clf)

Rodando Ablation de Classificação (3 sementes por configuração)...


,Experimento,Params,F1_macro_media,F1_macro_std,Acc_media,ROC_AUC_media,Tempo_medio_s
0,Ativ: Tanh,2148,0.766412,0.011042,0.765556,0.940425,1.846698
1,Ativ: GELU,2148,0.761201,0.004171,0.762222,0.931167,1.990506
2,"Arq: [128, 64, 32]",14308,0.760550,0.009445,0.758889,0.937044,1.674837
3,Dropout: 0.4,2148,0.758227,0.005532,0.758889,0.936501,3.155841
4,Opt: SGD (lr=1e-2),2148,0.749031,0.009448,0.748889,0.934834,1.156944
5,Dropout: 0.2,2148,0.745536,0.006706,0.746667,0.931619,2.326677
6,Arq: [16],548,0.742733,0.008840,0.745556,0.928210,3.103865
7,"Baseline [32,32] ReLU Adam 1e-3",2148,0.740447,0.015775,0.741111,0.928371,2.723045
8,"Arq: [64, 64]",6340,0.739829,0.004895,0.740000,0.928884,1.645391
9,LR: 1e-4 (Adam),2148,0.736712,0.010286,0.737778,0.927768,8.083050
